In [1]:
%load_ext line_profiler

In [2]:
time = %timeit -o -r1 -n4 x = [i for i in range(100000) ]

2 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 4 loops each)


In [3]:
print(time)

2 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 4 loops each)


In [4]:
def convert_units(heroes, heights, weights):
    new_hts = [ht * 0.39370 for ht in heights]
    new_wts = [wt * 2.20462 for wt in weights]
    hero_data = {}
    for i,hero in enumerate(heroes):
        hero_data[hero] = (new_hts[i], new_wts[i])
    return hero_data

heroes= ['a','b','c']

heights= [4,3.5,6.2]

weights= [9.1,4.8,5]

convert_units(heroes, heights, weights)

{'a': (1.5748, 20.062041999999998),
 'b': (1.37795, 10.582175999999999),
 'c': (2.44094, 11.0231)}

In [5]:
%lprun -f convert_units convert_units(heroes, heights, weights)

Timer unit: 1e-09 s

Total time: 1.0873e-05 s
File: /tmp/ipykernel_26980/1288718352.py
Function: convert_units at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def convert_units(heroes, heights, weights):
     2         1       2864.0   2864.0     26.3      new_hts = [ht * 0.39370 for ht in heights]
     3         1       1192.0   1192.0     11.0      new_wts = [wt * 2.20462 for wt in weights]
     4         1        503.0    503.0      4.6      hero_data = {}
     5         4       2800.0    700.0     25.8      for i,hero in enumerate(heroes):
     6         3       1671.0    557.0     15.4          hero_data[hero] = (new_hts[i], new_wts[i])
     7         1       1843.0   1843.0     17.0      return hero_data

## COMPARING TWO FUNCTIONS THAT COUNT ERRORS IN A SERVER LOGGING FILE

In [6]:
from streaming_functions import read_large_file_count , count_errors,read_large_file
import tracemalloc

### standard approach 

In [7]:
%lprun -f read_large_file_count read_large_file_count('server.log')

Timer unit: 1e-09 s

Total time: 0.00208633 s
File: /home/lgmada/DataScienceProjects/learning_python/streaming_functions.py
Function: read_large_file_count at line 11

Line #      Hits         Time  Per Hit   % Time  Line Contents
    11                                           def read_large_file_count(path):
    12         1        744.0    744.0      0.0      errors = 0
    13         2     589356.0 294678.0     28.2      with open(path, "r") as f:
    14      1249     800321.0    640.8     38.4          for line in f:            # file objects are already iterators!
    15      1248     357959.0    286.8     17.2              if "ERROR" in line :
    16       468     132913.0    284.0      6.4                  errors += 1
    17                                                       else:
    18       780     204704.0    262.4      9.8                  continue
    19         1        335.0    335.0      0.0          return errors

### streaming approach with generators

In [8]:
%lprun -f count_errors count_errors("server.log")

Timer unit: 1e-09 s

Total time: 0.00100083 s
File: /home/lgmada/DataScienceProjects/learning_python/streaming_functions.py
Function: count_errors at line 6

Line #      Hits         Time  Per Hit   % Time  Line Contents
     6                                           def count_errors(path):
     7         1    1000829.0    1e+06    100.0      return sum(1 for line in read_large_file(path) if "ERROR" in line)

In [9]:
# Test Standard Approach
tracemalloc.start()
read_large_file_count("server.log")
_, peak_standard = tracemalloc.get_traced_memory()
tracemalloc.stop()

# Test Generator Approach
tracemalloc.start()
count_errors("server.log")
_, peak_generator = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Standard Peak:  {peak_standard / 1024:.2f} KB")
print(f"Generator Peak: {peak_generator / 1024:.2f} KB")

Standard Peak:  146.32 KB
Generator Peak: 147.21 KB
